# pi-metaboqc Interactive Pipeline Tutorial

This notebook provides a step-by-step interactive guide to the `pi-metaboqc` metabolomics data quality control pipeline. 

By executing each cell sequentially, you can inspect the intermediate data structures and quality assessment (QA) reports generated at each stage of the workflow.

[TOC]

## Step 0: Environment Setup

First, we will import the core modules of `pi-metaboqc`, configure the input data paths, and create a dedicated output directory for the results of this notebook.

In [1]:
# -------------------------------------------------------------------------
# Step 0: Environment Setup and Parameter Loading
# -------------------------------------------------------------------------
import os
import sys
import pandas as pd
from loguru import logger
from typing import Dict, Any, Tuple

# Append the src directory to system path to import pimqc modules
sys.path.append(os.path.abspath(os.path.join("..", "src")))

import pimqc.io_utils as iu
import pimqc.plot_utils as pu
from pimqc.dataset_builder import build_dataset
from pimqc.assessment import MetaboIntAssessor
from pimqc.filtering import MetaboIntFilter
from pimqc.correction import MetaboIntCorrector
from pimqc.imputation import MetaboIntImputer
from pimqc.normalization import MetaboIntNormalizer

# Define standard directories
DATA_DIR: str = os.path.join("..", "src", "pimqc", "data")
OUTPUT_DIR: str = os.path.join(".", "tutorial_output")
iu._check_dir_exists(dir_path=OUTPUT_DIR, handle="makedirs")

# Load SSOT pipeline parameters
PARAMS_PATH: str = os.path.join(DATA_DIR, "pipeline_parameters.json")
params: Dict[str, Any] = iu._load_json_file(input_file_path=PARAMS_PATH)

# Load raw matrices

# meta_df: pd.DataFrame = pd.read_csv(
#     os.path.join(DATA_DIR, "project_meta.csv"), header=[0])
meta_df: pd.DataFrame = pd.read_csv(
    os.path.join(DATA_DIR, "project_meta_with_simu_group.csv"), header=[0])

int_df: pd.DataFrame = pd.read_csv(
    os.path.join(DATA_DIR, "project_intensity.csv"), index_col=[0], header=[0])

logger.info("Environment initialized successfully.")

2026-04-09 19:32:55.133 | WARNING  | pimqc.io_utils:_check_dir_exists:207 - No such directory, creating a new directory:
	.\tutorial_output.
2026-04-09 19:32:55.161 | INFO     | __main__:<module>:41 - Environment initialized successfully.


## Step 1: Dataset Construction
We will merge the raw intensity dataframe with the metadata mapping to 
build the standardized `MetaboInt` base object. This multi-indexed dataframe 
ensures analytical features and sample metadata are strictly aligned.

In [2]:
# -------------------------------------------------------------------------
# Step 1: Build Standardized Dataset
# -------------------------------------------------------------------------
step1_dir: str = os.path.join(OUTPUT_DIR, "01_Raw_Data")

raw_data = build_dataset(
    meta_info=meta_df,
    int_df=int_df,
    pipeline_params=params,
    mode=params.get("MetaboInt", {}).get("mode", "POS"),
    batch=params.get("MetaboInt", {}).get("batch", "Batch"),
    sample_type=params.get(
        "MetaboInt", {}).get("sample_type", "Sample Type"),
    bio_group=params.get(
        "MetaboInt", {}).get("bio_group", "Bio Group"),
    sample_name=params.get(
        "MetaboInt", {}).get("sample_name", "Sample Name"),
    inject_order=params.get(
        "MetaboInt", {}).get("inject_order", "Inject Order"),
    output_dir=step1_dir
)

logger.info(f"Raw MetaboInt object built. Shape: {raw_data.shape}")

2026-04-09 19:32:55.173 | WARNING  | pimqc.io_utils:_check_dir_exists:207 - No such directory, creating a new directory:
	.\tutorial_output\01_Raw_Data.
2026-04-09 19:32:55.272 | SUCCESS  | pimqc.io_utils:time_wrap:221 - Execution time of "build_dataset": 00:00:00.102.
2026-04-09 19:32:55.273 | INFO     | __main__:<module>:23 - Raw MetaboInt object built. Shape: (376, 466)


## Step 2: Quality Assessment (Raw Data)
Before any mathematical transformation, we perform an initial QA to evaluate 
the baseline condition of the untargeted metabolomics data.

In [3]:
# -------------------------------------------------------------------------
# Step 2: Initial QA
# -------------------------------------------------------------------------
step2_dir: str = os.path.join(OUTPUT_DIR, "02_QA_Raw_Data")

qa_raw = MetaboIntAssessor(data=raw_data, pipeline_params=params)
qa_raw.execute_qa(output_dir=step2_dir)

logger.info("Initial QA completed.")

2026-04-09 19:32:55.279 | WARNING  | pimqc.io_utils:_check_dir_exists:207 - No such directory, creating a new directory:
	.\tutorial_output\02_QA_Raw_Data.
2026-04-09 19:32:59.785 | SUCCESS  | pimqc.io_utils:time_wrap:221 - Execution time of "execute_qa": 00:00:04.505.
2026-04-09 19:32:59.785 | INFO     | __main__:<module>:9 - Initial QA completed.


## Step 3: High Missing Value (MV) Feature Filter (Stage 1)
We drop features containing an unacceptably high proportion of missing 
values based on the defined `mv_group_tol` biological override rule.

In [4]:
# -------------------------------------------------------------------------
# Step 3: Stage 1 Filtering
# -------------------------------------------------------------------------
step3_dir: str = os.path.join(OUTPUT_DIR, "03_Filtered_Stage1")

fltr_stg1 = MetaboIntFilter(data=raw_data, pipeline_params=params)
filtered_data = fltr_stg1.execute_mv_fltr(output_dir=step3_dir)

logger.info(f"Post-Stage 1 data shape: {filtered_data.shape}")

2026-04-09 19:32:59.793 | INFO     | pimqc.filtering:execute_mv_fltr:132 - Stage 1 MV Filter: Using Biological Group level.
2026-04-09 19:32:59.803 | WARNING  | pimqc.io_utils:_check_dir_exists:207 - No such directory, creating a new directory:
	.\tutorial_output\03_Filtered_Stage1.
2026-04-09 19:33:00.545 | INFO     | pimqc.filtering:execute_mv_fltr:197 - Stage 1 filter completed. Outputs saved to .\tutorial_output\03_Filtered_Stage1
2026-04-09 19:33:00.546 | SUCCESS  | pimqc.io_utils:time_wrap:221 - Execution time of "execute_mv_fltr": 00:00:00.753.
2026-04-09 19:33:00.546 | INFO     | __main__:<module>:9 - Post-Stage 1 data shape: (347, 466)


## Step 4: Signal Drift & Batch Effect Correction
Using Quality Control (QC) samples, we build regression models (e.g., LOESS, 
Random Forest, SVR) to estimate and correct analytical signal drift 
across injection sequences.

In [5]:
# -------------------------------------------------------------------------
# Step 4: Signal Correction
# -------------------------------------------------------------------------
step4_dir: str = os.path.join(OUTPUT_DIR, "04_Corrected_Data")

sc_engine = MetaboIntCorrector(data=filtered_data, pipeline_params=params)
intra_data, inter_data = sc_engine.execute_sc(output_dir=step4_dir)

logger.info(f"Intra-batch SC shape: {intra_data.shape}")
logger.info(f"Inter-batch SC shape: {inter_data.shape}")

2026-04-09 19:33:00.553 | WARNING  | pimqc.io_utils:_check_dir_exists:207 - No such directory, creating a new directory:
	.\tutorial_output\04_Corrected_Data.
2026-04-09 19:33:00.561 | INFO     | pimqc.correction:process_matrix_fit:401 - Using 3 threads for parallel SC analysis.
2026-04-09 19:33:00.562 | INFO     | pimqc.correction:process_matrix_fit:410 - Analyzing 3 batches concurrently using QC-RLSC.
SC [All Batches]: 100%|█████████████████████████████████| 1041/1041 [ETA: 00:00]
2026-04-09 19:33:05.901 | SUCCESS  | pimqc.io_utils:time_wrap:221 - Execution time of "execute_sc": 00:00:05.347.
2026-04-09 19:33:05.902 | INFO     | __main__:<module>:9 - Intra-batch SC shape: (347, 466)
2026-04-09 19:33:05.902 | INFO     | __main__:<module>:10 - Inter-batch SC shape: (347, 466)


## Step 5 & 6: Quality Assessment on Corrected Data
We assess the effectiveness of the signal correction algorithms by examining 
the intra-batch and inter-batch states separately.

In [6]:
# -------------------------------------------------------------------------
# Step 5 & 6: QA on Intra and Inter-batch Corrected Data
# -------------------------------------------------------------------------
step5_dir: str = os.path.join(OUTPUT_DIR, "05_QA_Intra_SC")
step6_dir: str = os.path.join(OUTPUT_DIR, "06_QA_Inter_SC")

qa_intra = MetaboIntAssessor(data=intra_data, pipeline_params=params)
qa_intra.execute_qa(output_dir=step5_dir)

qa_inter = MetaboIntAssessor(data=inter_data, pipeline_params=params)
qa_inter.execute_qa(output_dir=step6_dir)

logger.info("Post-Correction QA reports generated.")

2026-04-09 19:33:05.910 | WARNING  | pimqc.io_utils:_check_dir_exists:207 - No such directory, creating a new directory:
	.\tutorial_output\05_QA_Intra_SC.
2026-04-09 19:33:10.394 | SUCCESS  | pimqc.io_utils:time_wrap:221 - Execution time of "execute_qa": 00:00:04.484.
2026-04-09 19:33:10.395 | WARNING  | pimqc.io_utils:_check_dir_exists:207 - No such directory, creating a new directory:
	.\tutorial_output\06_QA_Inter_SC.
2026-04-09 19:33:14.813 | SUCCESS  | pimqc.io_utils:time_wrap:221 - Execution time of "execute_qa": 00:00:04.418.
2026-04-09 19:33:14.814 | INFO     | __main__:<module>:13 - Post-Correction QA reports generated.


## Step 7: Low Quality Feature Filter (Stage 2)
Features are evaluated against Blank sample ratios and QC Relative Standard 
Deviation (RSD). Missing indices are passed explicitly as current features 
default to MAR behavior prior to imputation.

In [7]:
# -------------------------------------------------------------------------
# Step 7: Stage 2 Filtering
# -------------------------------------------------------------------------
step7_dir: str = os.path.join(OUTPUT_DIR, "07_Filtered_Stage2")

fltr_stg2 = MetaboIntFilter(data=inter_data, pipeline_params=params)

idx_mar: pd.Index = inter_data.index
idx_mnar: pd.Index = pd.Index([])

quality_filtered_data = fltr_stg2.execute_quality_fltr(
    idx_mar=idx_mar, idx_mnar=idx_mnar, output_dir=step7_dir
)

logger.info(f"Post-Stage 2 data shape: {quality_filtered_data.shape}")

2026-04-09 19:33:14.823 | INFO     | pimqc.filtering:execute_quality_fltr:241 - Stage 2 Filter: Executing Blank Ratio Check.
2026-04-09 19:33:14.827 | INFO     | pimqc.filtering:execute_quality_fltr:261 - Stage 2 Filter: Executing QC RSD Check.
2026-04-09 19:33:14.833 | WARNING  | pimqc.io_utils:_check_dir_exists:207 - No such directory, creating a new directory:
	.\tutorial_output\07_Filtered_Stage2.
2026-04-09 19:33:15.940 | INFO     | pimqc.filtering:execute_quality_fltr:317 - Stage 2 filter completed. Outputs saved to .\tutorial_output\07_Filtered_Stage2
2026-04-09 19:33:15.941 | SUCCESS  | pimqc.io_utils:time_wrap:221 - Execution time of "execute_quality_fltr": 00:00:01.118.
2026-04-09 19:33:15.941 | INFO     | __main__:<module>:15 - Post-Stage 2 data shape: (166, 466)


## Step 8: Quality Assessment on Filtered Data

In [8]:
# -------------------------------------------------------------------------
# Step 8: QA Post-Filtering
# -------------------------------------------------------------------------
step8_dir: str = os.path.join(OUTPUT_DIR, "08_QA_Filtered_Data")
 
qa_fltr = MetaboIntAssessor(data=quality_filtered_data, pipeline_params=params)
qa_fltr.execute_qa(output_dir=step8_dir)

2026-04-09 19:33:15.948 | WARNING  | pimqc.io_utils:_check_dir_exists:207 - No such directory, creating a new directory:
	.\tutorial_output\08_QA_Filtered_Data.
2026-04-09 19:33:20.340 | SUCCESS  | pimqc.io_utils:time_wrap:221 - Execution time of "execute_qa": 00:00:04.392.


## Step 9: Missing Value Imputation
We address remaining structural missing values using probabilistic logic, 
Random Forest, or traditional k-Nearest Neighbors (KNN).

In [9]:
# -------------------------------------------------------------------------
# Step 9: Missing Value Imputation & Evaluation
# -------------------------------------------------------------------------

step9_dir = os.path.join(OUTPUT_DIR, "09_Imputed_Data")

# 1. Initialize imputation engine
imp_engine = MetaboIntImputer(
    quality_filtered_data, 
    pipeline_params=params
)

# 2. Smart Routing: Intercept 'auto' mode to prevent duplicate benchmarking
target_method = imp_engine.attrs.get("method", "auto")
if target_method.lower() == "auto":
    
    # Trigger the benchmark once and lock in the optimal method
    target_method = imp_engine._auto_select_method(mask_ratio=0.05)
    
    # Overwrite the 'auto' attribute to prevent downstream re-triggering
    imp_engine.attrs["method"] = target_method 

# 3. Execute Simulation (Calculates NRMSE and silently saves scatter plot)
sim_nrmse = imp_engine.execute_nrmse_simulation(
    method=target_method, output_dir=step9_dir
)

# 4. Execute Imputation (Silently saves imputed CSV, KDE, and PCA plots)
imputed_data = imp_engine.execute_imputation(
    method=target_method, output_dir=step9_dir
)

logger.success(
    f"Imputation completed using '{target_method}'. Shape: {imputed_data.shape}. "
    f"All data and analytical plots saved to {step9_dir}."
)

2026-04-09 19:33:20.349 | INFO     | pimqc.imputation:_auto_select_method:244 - Running 'auto' mode: Benchmarking imputation algorithms...
2026-04-09 19:33:20.522 | INFO     | pimqc.imputation:_auto_select_method:252 -   - Algorithm [probabilistic]: NRMSE = 0.4660
2026-04-09 19:33:20.593 | INFO     | pimqc.imputation:_auto_select_method:252 -   - Algorithm [knn          ]: NRMSE = 0.0462
2026-04-09 19:33:20.600 | INFO     | pimqc.imputation:_auto_select_method:252 -   - Algorithm [halfmin      ]: NRMSE = 0.6972
2026-04-09 19:33:20.605 | INFO     | pimqc.imputation:_auto_select_method:252 -   - Algorithm [min          ]: NRMSE = 0.6546
2026-04-09 19:33:20.605 | SUCCESS  | pimqc.imputation:_auto_select_method:260 - Auto-selection complete. Optimal algorithm: 'knn'
2026-04-09 19:33:20.606 | INFO     | pimqc.imputation:execute_nrmse_simulation:280 - Executing NRMSE simulation for 'knn' (Mask ratio: 0.05)...
2026-04-09 19:33:20.671 | WARNING  | pimqc.io_utils:_check_dir_exists:207 - No such

## Step 10: Quality Assessment on Imputed Data

In [10]:
# -------------------------------------------------------------------------
# Step 10: QA Post-Imputation
# -------------------------------------------------------------------------
step10_dir: str = os.path.join(OUTPUT_DIR, "10_QA_Imputed_Data")

qa_imp = MetaboIntAssessor(data=imputed_data, pipeline_params=params)
qa_imp.execute_qa(output_dir=step10_dir)

2026-04-09 19:33:23.410 | WARNING  | pimqc.io_utils:_check_dir_exists:207 - No such directory, creating a new directory:
	.\tutorial_output\10_QA_Imputed_Data.
2026-04-09 19:33:27.881 | SUCCESS  | pimqc.io_utils:time_wrap:221 - Execution time of "execute_qa": 00:00:04.470.


## Step 11: Data Normalization
Features are scaled and normalized (e.g., PQN, VSN) to remove remaining 
systematic bias and stabilize variance across the detection range.

In [11]:
# -------------------------------------------------------------------------
# Step 11: Normalization
# -------------------------------------------------------------------------
step11_dir: str = os.path.join(OUTPUT_DIR, "11_Normalized_Data")

# 1. Initialize normalization engine (norm_engine serves as the Raw Data)
norm_engine = MetaboIntNormalizer(imputed_data, pipeline_params=params)
# 2. Execute normalization (this automatically saves comparison PDFs to disk)
normalized_data = norm_engine.execute_norm(output_dir=step11_dir)

logger.info(f"Normalized dataset shape: {normalized_data.shape}")

2026-04-09 19:33:27.888 | WARNING  | pimqc.io_utils:_check_dir_exists:207 - No such directory, creating a new directory:
	.\tutorial_output\11_Normalized_Data.
2026-04-09 19:33:27.889 | INFO     | pimqc.normalization:execute_norm:350 - Permanently dropping 24 Blank samples.
2026-04-09 19:33:27.890 | INFO     | pimqc.normalization:execute_norm:361 - Applying Normalization | Col: PQN | Row: VSN
2026-04-09 19:33:54.094 | SUCCESS  | pimqc.io_utils:time_wrap:221 - Execution time of "execute_norm": 00:00:26.206.
2026-04-09 19:33:54.095 | INFO     | __main__:<module>:11 - Normalized dataset shape: (166, 442)


## Step 12: Final Quality Assessment
The fully processed and refined dataset is evaluated one last time, 
confirming its readiness for downstream statistical learning and 
biomarker discovery.

In [12]:
# -------------------------------------------------------------------------
# Step 12: Final QA
# -------------------------------------------------------------------------
step12_dir: str = os.path.join(OUTPUT_DIR, "12_QA_Normalized_Data")

qa_final = MetaboIntAssessor(data=normalized_data, pipeline_params=params)
qa_final.execute_qa(output_dir=step12_dir)

logger.success("Pipeline tutorial completed successfully.")

2026-04-09 19:33:54.101 | WARNING  | pimqc.io_utils:_check_dir_exists:207 - No such directory, creating a new directory:
	.\tutorial_output\12_QA_Normalized_Data.
2026-04-09 19:33:58.707 | SUCCESS  | pimqc.io_utils:time_wrap:221 - Execution time of "execute_qa": 00:00:04.606.
2026-04-09 19:33:58.708 | SUCCESS  | __main__:<module>:9 - Pipeline tutorial completed successfully.
